# 多模态选修：patchify → vision embedding → projector → 文本 Decoder 接口

In [ ]:
import torch

images = torch.arange(2 * 3 * 4 * 4.0).reshape(2, 3, 4, 4)
patches = images.unfold(2, 2, 2).unfold(3, 2, 2)
patches = patches.permute(0, 2, 3, 1, 4, 5).reshape(2, 4, 12)
projector = torch.nn.Linear(12, 16)
vision_tokens = projector(patches)
assert vision_tokens.shape == (2, 4, 16)
print(vision_tokens.shape)

不下载权重、不训练大型 VLM。完成 learning/labs/starter/21_multimodal_bridge.py 后运行独立核查。

## 选修边界与目标

主线严格是文本 LLM。本选修只建立接口直觉：`image → patches → vision embeddings → projector/resampler → text decoder`；不下载权重、不训练大型 VLM，也不把玩具结果外推为视觉语言能力。

In [ ]:
import torch

images = torch.arange(2 * 3 * 8 * 8, dtype=torch.float32).reshape(2, 3, 8, 8)
patch = 4
assert images.shape[-1] % patch == 0 and images.shape[-2] % patch == 0
patches = (
    images.unfold(2, patch, patch)
    .unfold(3, patch, patch)
    .permute(0, 2, 3, 1, 4, 5)
    .reshape(2, 4, 3 * patch * patch)
)
print("patch tensor:", tuple(patches.shape))

## 1. Projector 与 resampler

线性/MLP projector 只对齐通道维，token 数不变；resampler 用可学习查询压缩视觉 token；cross-attention 让文本查询读取独立视觉 memory。三者数据流和成本不同。

In [ ]:
vision_width, text_width = patches.shape[-1], 32
projector = torch.nn.Sequential(
    torch.nn.Linear(vision_width, 64), torch.nn.GELU(), torch.nn.Linear(64, text_width)
)
vision_tokens = projector(patches)
assert vision_tokens.shape == (2, 4, text_width)
print("decoder-compatible:", tuple(vision_tokens.shape))

## 2. 拼接 token 仍需要显式协议

需定义 `<image>` 边界、attention 权限、label mask、位置编码与多图顺序。视觉占位通常不参与文本 next-token loss；能拼接形状不等于学会 grounding。

In [ ]:
text_tokens = torch.randn(2, 5, text_width)
sequence = torch.cat([vision_tokens, text_tokens], dim=1)
loss_mask = torch.cat(
    [torch.zeros(2, 4, dtype=torch.bool), torch.ones(2, 5, dtype=torch.bool)], dim=1
)
assert sequence.shape[:2] == loss_mask.shape
print({"joint_shape": tuple(sequence.shape), "supervised_tokens": loss_mask.sum(1).tolist()})

## 3. 视觉位置与 M-RoPE

图像 patch 同时有行、列坐标；视频还有时间轴。M-RoPE 类方案把旋转维度分给时间/高度/宽度，具体分配与文本位置连续性属于模型协议。

In [ ]:
grid_h = grid_w = 2
coords = torch.tensor([(r, c) for r in range(grid_h) for c in range(grid_w)])
assert coords.shape == (4, 2)
print("[row,col]：", coords.tolist())

## 4. 数据与评测风险

图文对可能错配；文本先验会造成看图胡说。分别评估感知、OCR、空间关系、知识与幻觉，并用图像扰动/遮挡检查回答是否真的依赖视觉。

## 练习与来源

完成 `../../learning/labs/starter/21_multimodal_bridge.py`。来源：[ViT](https://arxiv.org/abs/2010.11929)、[CLIP](https://arxiv.org/abs/2103.00020)、[Flamingo](https://arxiv.org/abs/2204.14198)、[LLaVA](https://arxiv.org/abs/2304.08485)。互动图见 `../../interactive/multimodal_flow.html`。

## 完成断言

- [ ] 手算 patch 数；[ ] projector 输出等于文本 `d_model`；[ ] 视觉 token 不误计文本 loss；[ ] 区分 projector/resampler/cross-attention；[ ] 明确接口不等于训练完成的 VLM。